# Exercise — Module 8: Time-Series Forecasting of Reference Drift
#
**Dataset (โปรดอ่านก่อนใช้งาน):**
- *NOAA GML globally-averaged atmospheric CO₂, monthly means* — Lan, X., Thoning, K.W., Dlugokencky, E.J.: *Trends in globally-averaged CO₂ determined from NOAA Global Monitoring Laboratory measurements*, Version 2026-09, https://gml.noaa.gov/ccgg/trends/ (U.S. Government work; fair credit required)
- ไฟล์: `datasets/co2_mm_gl.csv` — header `#` + sentinel `-99.99` (เดือนที่วัดไม่ได้)
- แมปเข้าเคสเมโทรโลยี: `decimal` → เวลานับจาก calibration ล่าสุด · `average` → ค่าเบี่ยงเบนของ working standard (ppm)
#
ทำในโน้ตบุ๊กนี้ได้เลย — รันเซลล์เตรียมข้อมูลก่อน แล้วเติมโค้ดในช่อง `# TODO`

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~60 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)


In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise FileNotFoundError("ไม่พบ datasets/ — รันโน้ตบุ๊กจากใน repo")


DATA = data_dir()

## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)

ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)

design matrix 2 lags (จาก book)

In [2]:
import polars as pl  # noqa: E402
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

raw = pl.read_csv(DATA / "co2_mm_gl.csv", comment_prefix="#")
df = raw.with_columns(
    pl.when(pl.col("average") < 0)
    .then(None)
    .otherwise(pl.col("average"))
    .alias("average")
).drop_nulls("average")
v = df["average"].to_numpy()
dec = df["decimal"].to_numpy()
month = df["month"].to_numpy()
N = len(v)
print("จำนวนเดือน:", N, "| ช่วงเวลา:", dec[0], "→", dec[-1])

X = np.column_stack([v[12 : N - 1], v[0 : N - 13]])
yv = v[13:]

จำนวนเดือน: 570 | ช่วงเวลา: 1979.042 → 2026.458


## ข้อ A1 — โปรไฟล์ฤดูกาลของ "ห้องแล็บ" ด้วยมือของคุณเอง
#
ใน book เราถอด seasonal profile ด้วย rolling mean หน้าต่าง 12 เดือน คราวนี้ให้คุณคำนวณเองอีกวิธี: **trend จาก rolling mean แบบ trailing** (ใช้ข้อมูลย้อนหลังอย่างเดียว — สมจริงกว่า เพราะขณะวัดจริงเราไม่รู้อนาคต) แล้วหา profile รายเดือน และตอบว่าเดือนใดสูงสุด/ต่ำสุด amplitude เท่าไร (เทียบ book: พ.ค. สูงสุด, ก.ย.–ต.ค. ต่ำสุด, amplitude ≈ 4.4 ppm)

TODO: trend แบบ trailing — rolling_mean(window_size=12, center=False) หรือคำนวณเอง
แล้วหา detrended = s - trend, profile รายเดือน, amplitude

คำถาม: profile ต่างจากแบบ centered ไหม? เพราะอะไร (เกี่ยวกับ trend ที่ไต่ตลอด)

**คำใบ้:** `s.rolling_mean(window_size=12)` ค่า 11 จุดแรกจะเป็น null — drop ก่อนคำนวณ profile ไม่งั้น NaN กินทุกค่าเฉลี่ย ส่วนคำตอบ "ต่างไหม": detrended แบบ trailing ต่ำกว่าจริงในช่วงที่ trend ไต่ ลองพลอต profile สองแบบซ้อนกันดู

## ข้อ A2 — เพิ่ม lag 24 แล้วโมเดลดีขึ้นจริงไหม
#
ใน book ใช้ lags (1, 12) ได้ mean MAE **0.789** สร้าง design matrix 3 lags (1, 12, 24) เทียบกันด้วย TimeSeriesSplit แบบเดียวกัน — และอย่าลืมว่าเพิ่ม lag = เสียแถวหัว

TODO: X3 = column_stack ของ y_{t-1}, y_{t-12}, y_{t-24} และ y3 = y_t
วน TimeSeriesSplit(n_splits=5) กับ LinearRegression แล้วพิมพ์ MAE ต่อ fold + ค่าเฉลี่ย
คำถาม: ดีขึ้นไหม? lag 24 เพิ่มข้อมูลอะไรที่ lag 12 ให้ไม่ได้ (ในซีรีส์นี้)

In [3]:
from sklearn.linear_model import LinearRegression  # noqa: E402, F401
from sklearn.model_selection import TimeSeriesSplit  # noqa: E402, F401
from sklearn.metrics import mean_absolute_error  # noqa: E402, F401

**คำใบ้:** แถวแรกที่ใช้ได้คือ t = 25 → `X3 = np.column_stack([v[24:N-1], v[12:N-13], v[0:N-25]])`, `y3 = v[25:]` ค่าที่ควรได้ประมาณ 0.80–0.81 — แย่กว่า 2 lags เล็กน้อย

## ข้อ B1 — ทำ TimeSeriesSplit ให้เข้มขึ้นด้วย gap
#
โมเดลเรากิน lag 12 — fold ที่ test ต่อจาก train ทันที ทำให้ feature ของ test จุดแรก ๆ โผล่ใน train ด้วย (lag 12 ของ test จุดแรก = train จุดสุดท้าย) ให้ตัดช่องว่าง 12 จุดระหว่าง train/test ทุก fold (gap) แล้วเทียบ MAE กับแบบไม่มี gap

TODO: วน fold ด้วย tscv.split(X) แต่ตัด train ให้เหลือเฉพาะ index < te.min() - 12
แล้ววัด MAE เทียบกับ 0.789 (ไม่มี gap)

**คำใบ้:** ภายในลูป: `tr2 = tr[tr < te.min() - 12]` แล้ว fit กับ `X[tr2]` — ค่าที่ควรได้ ~0.80–0.82 (เข้มขึ้น ตัวเลขแย่ลงนิดเดียว — โมเดลไม่ได้โกงจาก lag ตัดผ่านขอบ fold)

## ข้อ B2 — XGBoost บน differences: ลองความลึกและจำนวนต้น
#
ใน book ใช้ XGBoost บน Δy ได้ MAE ~0.19 ทดลองสองชุด: `max_depth` 1, 3, 6 และ `n_estimators` 100, 300, 900 — ทำตารางเล็ก ๆ แล้วตอบว่าต้นไม้ลึกช่วยไหมกับซีรีส์ที่ราบเรียบแบบนี้

TODO: วนซ้อนสองลูป (depth × n_estimators) คำนวณ mean MAE แบบ reconstruct ระดับ
เหมือน book (pred = v[12 + te] + predicted Δ) แล้วพิมพ์ตาราง 3×3

In [4]:
from xgboost import XGBRegressor  # noqa: E402, F401

dy = v[12:] - v[: N - 12]
i0, i1 = 12, len(dy)
Xd = np.column_stack([dy[i0 - 1: i1 - 1], dy[i0 - 12: i1 - 12]])
yd = dy[i0:]

**คำใบ้:** เก็บผลใน dict หรือตาราง polars ก็ได้ — คาดหวังว่า depth=6 จะ **แย่ลง** (overfit Δ ที่มี noise เล็ก ๆ) — ถ้าเห็นตามนั้นให้อธิบายได้ว่าทำไม

## ข้อ C1 — พยากรณ์ 2 ปีด้วย recursive + วาดช่วงความไม่แน่นอน
#
สร้าง recursive forecast 24 เดือนจาก linear model (lags 1, 12) พร้อมแถบ ±2σ·√h แล้วตอบ: ที่ +24 เดือน ค่ากลางคือเท่าไร ขอบบนของช่วง ~95% เท่าไร ถ้า tolerance limit ของ "มาตรฐาน" อยู่ที่ +8 ppm จากค่าวันนี้ ควรตั้ง recalibration interval กี่เดือน

TODO: fit LinearRegression บน X, yv เต็มชุด
วน 24 ก้าว: x = [hist[-1], hist[-12]] → predict → append
คำนวณ sigma จาก residual แล้วพลอต: ข้อมูล 180 เดือนท้าย + เส้นพยากรณ์ + fill_between แถบ

**คำใบ้:** แถบโตตามระยะ: `2 * sigma * np.sqrt(np.arange(1, 25))` — ค่าที่ควรได้: กลาง ~431.6 ppm, ขอบบน ~440.7 ppm ที่ +24 เดือน

## ข้อ D1 — โจทย์คิด (ตอบใน markdown)
#
หัวหน้าขอคุณถาม: "ทำไมไม่ใช้ MAE จาก TimeSeriesSplit fold สุดท้าย fold เดียวเป็นตัวเลขรายงาน — มันทันสมัยที่สุดไม่ใช่หรือ" ตอบมา 3–4 บรรทัด โดยผูกกับ (1) ความผันผวนของ fold-level MAE ที่คุณเห็นในข้อ A2/B1 และ (2) การตัดสินใจ recalibration interval ที่อยู่บนตัวเลขนี้

คำตอบข้อ D1: